In [3]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import os
import zipfile

In [4]:
data_folder = None
zip_file = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'black pepper' in dirs and 'turmeric' in dirs:
        data_folder = root
        break
    for file_name in files:
        if file_name.lower().replace('-', '_') == 'spice_spectrum.zip':
            zip_file = os.path.join(root, file_name)

if data_folder is None and zip_file is not None:
    data_folder = '/kaggle/working/spice_spectrum'
    os.makedirs(data_folder, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as zip_data:
        zip_data.extractall(data_folder)

if data_folder is None:
    raise FileNotFoundError('By using Kaggle Add Input connect Spice_Spectrum.zip dataset')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Dataset folder:', data_folder)

Device: cpu
Dataset folder: /kaggle/input/datasets/ratulrayhanrabby/ratul-data


In [5]:
class_names = sorted([
    folder for folder in os.listdir(data_folder)
    if os.path.isdir(os.path.join(data_folder, folder))
])

print('Classes:', class_names)
print('Number of classes:', len(class_names))
assert len(class_names) == 11, 'Dataset_Class=11'

tr_images, tr_targets = [], []
val_images, val_targets = [], []

np.random.seed(42)

for label, class_name in enumerate(class_names):
    class_folder = os.path.join(data_folder, class_name)
    image_files = sorted([
        os.path.join(class_folder, file_name)
        for file_name in os.listdir(class_folder)
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    np.random.shuffle(image_files)
    split_index = int(len(image_files) * 0.8)

    for i, image_path in enumerate(image_files):
        with Image.open(image_path) as image:
            image = np.array(image.convert('RGB').resize((32, 32)))

        if i < split_index:
            tr_images.append(image)
            tr_targets.append(label)
        else:
            val_images.append(image)
            val_targets.append(label)

tr_images = np.array(tr_images, dtype=np.uint8)
tr_targets = np.array(tr_targets, dtype=np.int64)
val_images = np.array(val_images, dtype=np.uint8)
val_targets = np.array(val_targets, dtype=np.int64)

print('Training image shape:', tr_images.shape)
print('Validation image shape:', val_images.shape)
print('Training labels:', tr_targets.shape)
print('Validation labels:', val_targets.shape)

Classes: ['black pepper', 'cardamom', 'cinnamon', 'cloves', 'coriander', 'cumin', 'ginger', 'nutmeg', 'paprika', 'saffron', 'turmeric']
Number of classes: 11
Training image shape: (8800, 32, 32, 3)
Validation image shape: (2200, 32, 32, 3)
Training labels: (8800,)
Validation labels: (2200,)


In [6]:
batch_size_no = 32
learning_rate = 1e-3
epoch_number = 30

In [7]:
class SpiceSpectrum(Dataset):
    def __init__(self, x, y):
        x = torch.tensor(x)     # N, H, W, C
        x = x.float() / 255     # Normalization: 0..255 -> 0..1
        x = x.permute(0, 3, 1, 2)  # N, C, H, W
        x = x.reshape(-1, 3, 32, 32)

        y = torch.tensor(y)
        self.x, self.y = x, y

    def __getitem__(self, ix):
        x, y = self.x[ix], self.y[ix]
        return x.to(device), y.to(device)

    def __len__(self):
        return len(self.x)


def get_data():
    train = SpiceSpectrum(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=batch_size_no, shuffle=True)
    val = SpiceSpectrum(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=True)
    return trn_dl, val_dl

In [8]:
trn_dl, val_dl = get_data()
print('Number of training batches:', len(trn_dl))
print('Number of validation batches:', len(val_dl))

Number of training batches: 275
Number of validation batches: 1


In [9]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.activation import ReLU
from torch.optim import SGD, Adam

def get_model():
    model = nn.Sequential(
        # Feature Extractor / Convolution layers
        # 1st conv
        nn.Conv2d(3, 40, 3, padding=1), # 32x32
        nn.ReLU(),                       # 32x32
        nn.MaxPool2d(2),                 # 16x16
        # 2nd conv
        nn.Conv2d(40, 65, 3, padding=1), # 16x16
        nn.ReLU(),                       # 16x16
        nn.MaxPool2d(2),                 # 8x8
        # 3rd conv
        nn.Conv2d(65, 70, 3, padding=1), # 8x8
        nn.ReLU(),                       # 8x8
        nn.MaxPool2d(2),                 # 4x4
        # Classifier / Neural Network
        nn.Flatten(),
        nn.Linear(70 * 4 * 4, 256),
        nn.ReLU(),
        nn.Linear(256, 11)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=learning_rate)
    return model, loss_fn, optimizer

In [10]:
def train_one_epoch(data_loader, model, loss_fn, optimizer):
    model.train()
    epoch_losses, epoch_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        prediction = model(x)
        batch_loss = loss_fn(prediction, y)
        batch_loss.backward()
        optimizer.step()

        epoch_losses.append(batch_loss.item())
        is_correct = (prediction.argmax(-1) == y).tolist()
        epoch_accuracies.extend(is_correct)
    return np.mean(epoch_losses), np.mean(epoch_accuracies)

In [11]:
@torch.no_grad()
def validation(data_loader, model, loss_fn):
    model.eval()
    val_losses, val_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        prediction = model(x)
        val_losses.append(loss_fn(prediction, y).item())
        is_correct = (prediction.argmax(-1) == y).tolist()
        val_accuracies.extend(is_correct)
    return np.mean(val_losses), np.mean(val_accuracies)

In [12]:
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Trainable parameters: 355408


In [13]:
from torch import optim
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    factor=0.5,
    patience=0,
    threshold=0.001,
    min_lr=1e-5,
    threshold_mode='abs'
)

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(epoch_number):
    epoch_loss, epoch_accuracy = train_one_epoch(trn_dl, model, loss_fn, optimizer)
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    v_loss, v_acc = validation(val_dl, model, loss_fn)
    val_losses.append(v_loss)
    val_accuracies.append(v_acc)

    scheduler.step(v_loss)

    print(f'Epoch [{epoch + 1}/{epoch_number}]'
          f' | Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_accuracy*100:.2f}%'
          f' | Val Loss: {v_loss:.4f} | Val Accuracy: {v_acc*100:.2f}%')

Epoch [1/30] | Train Loss: 2.1129 | Train Accuracy: 23.23% | Val Loss: 1.8279 | Val Accuracy: 35.18%
Epoch [2/30] | Train Loss: 1.5302 | Train Accuracy: 46.80% | Val Loss: 1.3712 | Val Accuracy: 54.55%
Epoch [3/30] | Train Loss: 1.2139 | Train Accuracy: 58.44% | Val Loss: 1.2251 | Val Accuracy: 56.68%
Epoch [4/30] | Train Loss: 1.0076 | Train Accuracy: 66.17% | Val Loss: 0.9405 | Val Accuracy: 67.86%
Epoch [5/30] | Train Loss: 0.8240 | Train Accuracy: 72.59% | Val Loss: 0.8310 | Val Accuracy: 72.32%
Epoch [6/30] | Train Loss: 0.6880 | Train Accuracy: 77.18% | Val Loss: 0.7335 | Val Accuracy: 76.45%
Epoch [7/30] | Train Loss: 0.5786 | Train Accuracy: 80.34% | Val Loss: 0.7580 | Val Accuracy: 75.64%
Epoch [8/30] | Train Loss: 0.4045 | Train Accuracy: 86.53% | Val Loss: 0.5750 | Val Accuracy: 81.32%
Epoch [9/30] | Train Loss: 0.3559 | Train Accuracy: 88.18% | Val Loss: 0.5753 | Val Accuracy: 80.95%
Epoch [10/30] | Train Loss: 0.2747 | Train Accuracy: 91.42% | Val Loss: 0.5401 | Val Accura